# Thai Plate Synth — Colab training driver

Weekend-2 run: render ~5k synthetic plates, train YOLOv8n, evaluate.

**Runtime:** change to **GPU** (Runtime → Change runtime type → T4 GPU).

## 1. Clone repo + install

In [ ]:
!git clone https://github.com/simonyos/thai-plate-synth.git /content/thai-plate-synth
%cd /content/thai-plate-synth
!pip install -q -e .[train] ultralytics

## 2. Upload the font

`Sarun's ThangLuang` is not redistributed — download from <https://www.f0nt.com/release/saruns-thangluang/>, unzip, and upload the `.ttf` via the dialog below.

In [ ]:
from google.colab import files
import shutil
from pathlib import Path

uploaded = files.upload()
ttf = next((f for f in uploaded if f.lower().endswith('.ttf')), None)
assert ttf, 'upload a .ttf file'
dst = Path('assets/fonts/SarunsThangLuang.ttf')
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.move(ttf, dst)
print('font at', dst, dst.stat().st_size, 'bytes')

## 3. Render synth dataset (~5k plates, seed 0)

In [ ]:
import sys; sys.path.insert(0, '/content/thai-plate-synth/src')
!python -m thai_plate_synth.render --out data/synth_v1 --count 5000 --seed 0
!ls data/synth_v1/images/train | head; echo '---'; ls data/synth_v1/images/val | head

## 4. Train YOLOv8n on synth (~25 min on T4)

In [ ]:
!python -m thai_plate_synth.train --data data/synth_v1 --epochs 50 --imgsz 480 --batch 64 --name synth_v1

## 5. Inspect results

In [ ]:
from IPython.display import Image, display
run_dir = 'runs/detect/synth_v1'
!ls {run_dir}
for f in ('results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg'):
    p = f'{run_dir}/{f}'
    try: display(Image(p))
    except Exception as e: print('skip', p, e)

## 6. Download best weights

In [ ]:
from google.colab import files
files.download('runs/detect/synth_v1/weights/best.pt')